In [ ]:
# ============================================================
# Environment / package versions
# ============================================================
# Python   : 3.10.18
# pandas   : 2.3.2
# NumPy    : 2.1.2
# mofapy2  : 0.7.2
# mofax    : 0.3.7

In [1]:
from mofapy2.run.entry_point import entry_point
import pandas as pd
import numpy as np
import mofax as mfx
import sys
import os

# initialise the entry point
ent = entry_point()


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
       
 
        


In [3]:
model = "Paired"  #Choose from Unpaired, Paired and SCOT+
script_dir = os.getcwd()
OUTPUT_DIR = os.path.join(script_dir, "trained_model")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [19]:
allowed_models = {"Paired", "Unpaired", "SCOT+"}

if model not in allowed_models:
    print(f"Warning: invalid model '{model}'. Choose from: Paired, Unpaired, or SCOT+.")
    sys.exit()

print("Models correct!")

if model in ["Paired", "Unpaired"]:
    INPUT_DIR = os.path.join(script_dir, "processed_data")
    transcriptomics = pd.read_csv(os.path.join(INPUT_DIR, "Processed_transcriptomics.csv"))
    proteomics = pd.read_csv(os.path.join(INPUT_DIR, "Processed_proteomics.csv"))
    proteomics_mat = proteomics.set_index("Gene")
    transcriptomics_mat = transcriptomics.set_index("Gene")
    print("Proteomics matrix shape (features x samples):", proteomics_mat.shape)
    print("Transcriptomics matrix shape (features x samples):", transcriptomics_mat.shape)
elif model == "SCOT+":
    INPUT_DIR = os.path.join(script_dir, "aligned_data")
    transcriptomics = pd.read_csv(os.path.join(INPUT_DIR, "aligned_transcriptomics.csv"))
    proteomics = pd.read_csv(os.path.join(INPUT_DIR, "aligned_proteomics.csv"))
    proteomics_mat = proteomics.set_index("Gene")
    transcriptomics_mat = transcriptomics.set_index("Gene")
    print("Proteomics matrix shape (features x samples):", proteomics_mat.shape)
    print("Transcriptomics matrix shape (features x samples):", transcriptomics_mat.shape)
if model == "Unpaired":
    proteomics_mat.columns = [f"{x}_prot" for x in proteomics_mat.columns]
    transcriptomics_mat.columns = [f"{x}_rna" for x in transcriptomics_mat.columns]

Models correct!
Proteomics matrix shape (features x samples): (1500, 68)
Transcriptomics matrix shape (features x samples): (1500, 68)


In [ ]:
prot_samples = pd.Index(proteomics_mat.columns)
rna_samples = pd.Index(transcriptomics_mat.columns)

if model in ["Paired", "Unpaired"]:
    # Identify paried samples
    common_samples = prot_samples.intersection(rna_samples)
    prot_only = prot_samples.difference(rna_samples)
    rna_only = rna_samples.difference(prot_samples)
    print(f"Proteomics samples: {len(prot_samples)}")
    print(f"Transcriptomics samples: {len(rna_samples)}")
    print(f"Common paired samples: {len(common_samples)}")
    print(f"Proteomics-only samples: {len(prot_only)}")
    print(f"Transcriptomics-only samples: {len(rna_only)}")
if model == "Paired":
    if len(common_samples) == 0:
        raise ValueError("No shared SampleID values found between proteomics and transcriptomics matrices.")
    common_samples = common_samples.sort_values()
    proteomics_mat = proteomics_mat.loc[:, common_samples]
    transcriptomics_mat = transcriptomics_mat.loc[:, common_samples]
    if not all(proteomics_mat.columns == transcriptomics_mat.columns):
        raise ValueError("Sample order mismatch between proteomics and transcriptomics after subsetting.")
    print("Paired reference matrices created successfully.")
    print(f"Proteomics matrix shape: {proteomics_mat.shape}")
    print(f"Transcriptomics matrix shape: {transcriptomics_mat.shape}")
elif model == "Unpaired":
    if len(common_samples) > 0:
        raise ValueError("Unexpected shared sample IDs remain after suffixing sample names.")
elif model == "SCOT+":
    print(f"Proteomics samples: {len(prot_samples)}")
    print(f"Transcriptomics samples: {len(rna_samples)}")
    if not prot_samples.equals(rna_samples):
        common_samples = prot_samples.intersection(rna_samples).sort_values()
        print("Warning: sample axes are not identical. Subsetting to common samples.")
        print(f"Common paired samples after SCOT+: {len(common_samples)}")
        if len(common_samples) == 0:
            raise ValueError("No shared SampleID values found between SCOT+-aligned transcriptomics and proteomics matrices.")
            proteomics_mat = proteomics_mat.loc[:, common_samples]
            transcriptomics_mat = transcriptomics_mat.loc[:, common_samples]
    else:
        common_samples = prot_samples
        print("SCOT+ pseudo-paired sample axes match exactly.")
    if not all(proteomics_mat.columns == transcriptomics_mat.columns):
        raise ValueError("Sample order mismatch between SCOT-aligned proteomics and transcriptomics.")
    print("Pseudo-paired SCOT+ matrices ready for MOFA+.")
    print(f"Proteomics matrix shape after checks: {proteomics_mat.shape}")
    print(f"Transcriptomics matrix shape after checks: {transcriptomics_mat.shape}")

In [ ]:
#Convert to long format
proteomics_long = (proteomics_mat.reset_index().melt(id_vars="Gene", var_name="sample", value_name="value").rename(columns={"Gene": "feature"}))
proteomics_long["view"] = "Proteomics"
proteomics_long["feature"] = proteomics_long["feature"].astype(str) + "_prot"
transcriptomics_long = (transcriptomics_mat.reset_index().melt(id_vars="Gene", var_name="sample", value_name="value").rename(columns={"Gene": "feature"}))
transcriptomics_long["view"] = "Transcriptomics"
transcriptomics_long["feature"] = transcriptomics_long["feature"].astype(str) + "_rna"
mofa_long = pd.concat([proteomics_long, transcriptomics_long], ignore_index=True)
print("\nMOFA long-format preview:")
print(mofa_long.head())
print("\nView counts:")
print(mofa_long["view"].value_counts())
print("\nUnique samples by view:")
print(mofa_long.groupby("view")["sample"].nunique())

In [ ]:
ent.set_data_df(mofa_long, likelihoods=["gaussian", "gaussian"])

In [ ]:
ent.set_model_options(factors = 5)

In [ ]:
ent.set_train_options(convergence_mode="slow", dropR2=0.001, gpu_mode=True, seed=1)

In [ ]:
ent.build()
ent.run()

In [ ]:
if model == "Paired":
    ent.save(outfile=os.path.join(OUTPUT_DIR, "Paired_model.hdf5"))
elif model == "Unpaired":
    ent.save(outfile=os.path.join(OUTPUT_DIR, "Unpaired_model.hdf5"))
elif model == "SCOT+":
    ent.save(outfile=os.path.join(OUTPUT_DIR, "SCOT+-aligned_model.hdf5"))